# DATA 266 - Lab 2 - Part 2: Fine-tuning Stable Diffusion 1.5 with LoRA
## Notebook 4 (extra experiments): can the final model be improved?

Notebooks 1-3 are the required pipeline. They gave a final model (LoRA rank 32) with a much lower KID than the base model (0.036 against 0.095) but a slightly lower CLIP score (32.1 against 32.8). This extra notebook tries to improve on that.

**What is tried**

- four more LoRA runs that change one thing at a time compared with run 2 (learning rate, rank, loss weighting, number of steps), and
- the guidance scale of the sampler (how strongly the prompt is followed).

**How the choice is made without using the report prompts.** The 20 fixed prompts are the ones reported in notebook 3, so they are *not* used here. Instead, 40 **validation prompts** are taken from test captions that notebook 3 does not use. The images are scored with KID (against the 83 test images, as in notebook 3) and CLIP score. One limit: the same 83 test images are the KID reference in both notebooks, so this is a fair selection but not a completely untouched test. This is stated in the report.

**Decision rule (written before any extra run was trained).** An extra run replaces run 2 as the final model only if, on the validation prompts,

1. its KID is lower than the KID of run 2 by more than the KID standard deviation, **and**
2. its CLIP score is not lower than the CLIP score of run 2.

A different guidance scale is adopted for all three models only if it raises the validation CLIP score of the final run without making its KID worse by more than the standard deviation.

Everything is written to `outputs/part2/extra/`. Nothing from notebooks 1-3 is changed or overwritten here.

In [ ]:
# Cell 1 - Reproducibility and environment information

import os
import sys
import random
import subprocess
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import torch

SEED = 266

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def library_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"


# Git commit. Git is not always on the PATH inside a notebook kernel, so a few usual places are tried
commit = ""
for git in ["git", r"C:\Program Files\Git\cmd\git.exe", os.path.expanduser(r"~\AppData\Local\Programs\Git\cmd\git.exe")]:
    try:
        result = subprocess.run([git, "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
        commit = result.stdout.strip()
        break
    except FileNotFoundError:
        continue

run_info = {
    "commit": commit or "no Git repository here",
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "diffusers": library_version("diffusers"),
    "peft": library_version("peft"),
    "transformers": library_version("transformers"),
    "torchmetrics": library_version("torchmetrics"),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "seed": SEED
}

print(run_info)

## 1. Settings

The extra runs use the same data, batch size, optimizer, seed and sampler as runs 1 and 2. Each one changes **one** thing compared with run 2 (rank 32, learning rate 1e-4, 3,000 steps):

| Extra run | What changes | Why it is tried |
|---|---|---|
| `extra1_lr5e-5` | learning rate 5e-5 (half) | a smaller update may keep more of the base model's prompt following |
| `extra2_r64` | rank 64 | more capacity for the style |
| `extra3_snr5` | Min-SNR loss weighting (`snr_gamma` 5) | a known way to make diffusion training more stable |
| `extra4_5000steps` | 5,000 steps (about 27 epochs) | more training for a closer match to the dataset |

In [ ]:
import json
import time
import shutil
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from safetensors import safe_open
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"

DATA_DIR = Path("../data/part2")
OUT_DIR = Path("../outputs/part2")
TRAIN_DIR = OUT_DIR / "data_prep" / "images" / "train"
TEST_DIR = OUT_DIR / "data_prep" / "images" / "test"
MAIN_RUN_DIR = OUT_DIR / "runs"
EXTRA_DIR = OUT_DIR / "extra"
EXTRA_RUN_DIR = EXTRA_DIR / "runs"
GEN_DIR = EXTRA_DIR / "generated"
METRIC_DIR = EXTRA_DIR / "metrics"
SCRIPT_DIR = Path("../scripts/part2").resolve()

for folder in [EXTRA_RUN_DIR, GEN_DIR, METRIC_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Settings shared by every run (the same as in notebook 2)
SETTINGS = {
    "resolution": 512,
    "train_batch_size": 4,
    "gradient_accumulation_steps": 1,
    "max_train_steps": 3000,
    "learning_rate": 1e-4,
    "lr_scheduler": "cosine",
    "lr_warmup_steps": 0,
    "mixed_precision": "fp16",
    "seed": SEED,
}

# Only the differences from run 2 are listed
EXTRA_RUNS = {
    "extra1_lr5e-5": {"rank": 32, "learning_rate": 5e-5},
    "extra2_r64": {"rank": 64},
    "extra3_snr5": {"rank": 32, "snr_gamma": 5.0},
    "extra4_5000steps": {"rank": 32, "max_train_steps": 5000},
}

MEMORY_LIMIT_GB = 15     # so that the runs also fit on a 16 GB GPU
RETRAIN = False          # False: reuse finished runs instead of training again

# Validation images
N_VALIDATION = 40
STEPS = 30
GUIDANCE = 7.5
SIZE = 512

print(json.dumps(SETTINGS, indent=2))
print("Extra runs:", json.dumps(EXTRA_RUNS))

## 2. Validation prompts

The in-domain prompts of notebook 3 (the 10 test captions in `prompts.json`) are left out. From the remaining test captions, 40 different ones are drawn with seed 266.

In [ ]:
fixed_prompts = json.loads((DATA_DIR / "prompts.json").read_text(encoding="utf-8"))
used_captions = {p["prompt"] for p in fixed_prompts}

test_captions = pd.read_csv(OUT_DIR / "data_prep" / "split_test.csv")
free = test_captions[~test_captions["caption"].isin(used_captions)].drop_duplicates("caption")
print(f"Test captions: {len(test_captions)} | not used in prompts.json: {len(free)}")

validation = free.sample(n=N_VALIDATION, random_state=SEED).reset_index(drop=True)
validation["id"] = [f"V{i + 1:02d}" for i in range(len(validation))]
validation.to_csv(METRIC_DIR / "validation_prompts.csv", index=False)

assert not set(validation["caption"]) & used_captions, "a validation prompt is also a reported prompt"
print(validation[["id", "caption"]].head(8).to_string(index=False))

## 3. Train the extra runs

The same wrapper and the same diffusers script as in notebook 2, with a 15 GB memory cap. A run that is already finished is not trained again (`RETRAIN = False`).

In [ ]:
def train(run_name, options):
    """Train one extra LoRA run and return its measurements."""
    settings = {**SETTINGS, **{k: v for k, v in options.items() if k in SETTINGS}}
    output_dir = (EXTRA_RUN_DIR / run_name).resolve()
    summary_file = output_dir / "summary.json"

    if summary_file.exists() and (output_dir / "pytorch_lora_weights.safetensors").exists() and not RETRAIN:
        print(f"{run_name}: finished run found, not training again")
        return json.loads(summary_file.read_text())

    command = [
        sys.executable, str(SCRIPT_DIR / "run_training.py"),
        "--memory_limit_gb", str(MEMORY_LIMIT_GB),
        "--summary", str(summary_file),
        "--",
        "--pretrained_model_name_or_path", BASE_MODEL,
        "--variant", "fp16",
        "--train_data_dir", str(TRAIN_DIR.resolve()),
        "--image_column", "image",
        "--caption_column", "text",
        "--resolution", str(settings["resolution"]),
        "--center_crop",
        "--random_flip",
        "--train_batch_size", str(settings["train_batch_size"]),
        "--gradient_accumulation_steps", str(settings["gradient_accumulation_steps"]),
        "--gradient_checkpointing",
        "--max_train_steps", str(settings["max_train_steps"]),
        "--learning_rate", str(settings["learning_rate"]),
        "--lr_scheduler", settings["lr_scheduler"],
        "--lr_warmup_steps", str(settings["lr_warmup_steps"]),
        "--mixed_precision", settings["mixed_precision"],
        "--seed", str(settings["seed"]),
        "--rank", str(options["rank"]),
        "--checkpointing_steps", "100000",       # no intermediate checkpoints
        "--dataloader_num_workers", "0",
        "--output_dir", str(output_dir),
    ]
    if "snr_gamma" in options:
        command += ["--snr_gamma", str(options["snr_gamma"])]

    shutil.rmtree(output_dir / "logs", ignore_errors=True)
    output_dir.mkdir(parents=True, exist_ok=True)
    log_file = output_dir / "training.log"
    print(f"{run_name}: training with {options} ...")
    start = time.time()
    with open(log_file, "w", encoding="utf-8") as log:
        completed = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT)
    print(f"{run_name}: finished in {(time.time() - start) / 60:.1f} minutes (exit code {completed.returncode})")

    if completed.returncode != 0:
        tail = log_file.read_text(encoding="utf-8", errors="replace").replace("\r", "\n").strip().split("\n")[-15:]
        print("\n".join(tail))
        raise RuntimeError(f"training of {run_name} failed, see {log_file}")

    lines = log_file.read_text(encoding="utf-8", errors="replace").replace("\r", "\n").split("\n")
    keep = [l for l in lines if l.strip() and "it/s" not in l and "s/it" not in l]
    log_file.write_text("\n".join(keep[-60:]), encoding="utf-8")

    return json.loads(summary_file.read_text())


summaries = {}
for run_name, options in EXTRA_RUNS.items():
    summaries[run_name] = train(run_name, options)

## 4. Training table

The same measurements as in notebook 2, for runs 1 and 2 (read from `training_table.csv`) and the extra runs.

In [ ]:
UNET_PARAMETERS = 859_520_964       # from the base model


def lora_parameter_count(path):
    file = safe_open(str(path / "pytorch_lora_weights.safetensors"), "pt")
    return sum(file.get_tensor(key).numel() for key in file.keys())


def final_loss(path):
    events = EventAccumulator(str(path / "logs" / "text2image-fine-tune"))
    events.Reload()
    return float(pd.Series([e.value for e in events.Scalars("train_loss")]).tail(100).mean())


n_train = len(list(TRAIN_DIR.glob("*.png")))
rows = []
for run_name, options in EXTRA_RUNS.items():
    settings = {**SETTINGS, **{k: v for k, v in options.items() if k in SETTINGS}}
    path = EXTRA_RUN_DIR / run_name
    summary = summaries[run_name]
    trainable = lora_parameter_count(path)
    rows.append({
        "run": run_name, "rank": options["rank"], "learning_rate": settings["learning_rate"],
        "snr_gamma": options.get("snr_gamma", "-"), "steps": settings["max_train_steps"],
        "epochs": round(settings["max_train_steps"] * settings["train_batch_size"] / n_train, 1),
        "trainable_parameters": trainable,
        "peak_gpu_memory_gb": summary["peak_memory_allocated_gb"],
        "training_minutes": round(summary["training_seconds"] / 60, 1),
        "gpu_hours": round(summary["training_seconds"] / 3600, 3),
        "final_loss": round(final_loss(path), 4),
    })

main_table = pd.read_csv(OUT_DIR / "metrics" / "training_table.csv")
main_rows = main_table.rename(columns={"final_loss": "final_loss"})[
    ["run", "rank", "learning_rate", "steps", "epochs", "trainable_parameters", "peak_gpu_memory_gb",
     "training_minutes", "gpu_hours", "final_loss"]].copy()
main_rows["snr_gamma"] = "-"

extra_training_table = pd.concat([main_rows, pd.DataFrame(rows)], ignore_index=True)[
    ["run", "rank", "learning_rate", "snr_gamma", "steps", "epochs", "trainable_parameters",
     "peak_gpu_memory_gb", "training_minutes", "gpu_hours", "final_loss"]]
extra_training_table.to_csv(METRIC_DIR / "extra_training_table.csv", index=False)
print(extra_training_table.to_string(index=False))

## 5. Generate and score the validation images

For each model: the 40 validation prompts, seed 0, 30 steps, guidance 7.5, with the same sampler for every model. The safety filter is off here, as in notebook 3, because this is only a model comparison.

- **CLIP score**: 100 x the cosine similarity between an image and its prompt (CLIP ViT-B/16). Higher is better.
- **KID**: the 40 generated images against the 83 test images. Lower is better.

In [ ]:
from diffusers import StableDiffusionPipeline
from transformers import CLIPModel, CLIPProcessor
from torchmetrics.image.kid import KernelInceptionDistance

pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, variant="fp16", safety_checker=None
).to("cuda")
pipe.set_progress_bar_config(disable=True)

CLIP_NAME = "openai/clip-vit-base-patch16"
clip = CLIPModel.from_pretrained(CLIP_NAME).to("cuda").eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_NAME)


def clip_scores(images, texts):
    """100 x cosine similarity between each image and its own prompt (at least 0)."""
    scores = []
    for start in range(0, len(images), 16):
        inputs = clip_processor(text=texts[start:start + 16], images=images[start:start + 16],
                                return_tensors="pt", padding=True, truncation=True, max_length=77).to("cuda")
        with torch.no_grad():
            output = clip(**inputs)
        scores += torch.diagonal(output.logits_per_image).float().clamp(min=0).cpu().tolist()
    return scores


def to_uint8_batch(images):
    return torch.stack([torch.from_numpy(np.asarray(i.convert("RGB").resize((SIZE, SIZE)))).permute(2, 0, 1) for i in images])


real = to_uint8_batch([Image.open(f) for f in sorted(TEST_DIR.glob("*.png"))])
print("Real images (test set):", len(real))


def lora_path(model):
    if model == "base":
        return None
    if model in EXTRA_RUNS:
        return str((EXTRA_RUN_DIR / model).resolve())
    return str((MAIN_RUN_DIR / model).resolve())


def generate_and_score(model, guidance, save=False):
    """Generate the validation images with one model and return its KID and CLIP score."""
    if lora_path(model):
        pipe.load_lora_weights(lora_path(model))
    images = []
    for prompt in validation["caption"]:
        generator = torch.Generator("cuda").manual_seed(0)
        images.append(pipe(prompt, num_inference_steps=STEPS, guidance_scale=guidance,
                           height=SIZE, width=SIZE, generator=generator).images[0])
    if lora_path(model):
        pipe.unload_lora_weights()

    if save:
        (GEN_DIR / model).mkdir(parents=True, exist_ok=True)
        for image, prompt_id in zip(images, validation["id"]):
            image.save(GEN_DIR / model / f"{prompt_id}_s0.png")

    torch.manual_seed(SEED)
    kid = KernelInceptionDistance(subset_size=len(images), subsets=100, normalize=False).to("cuda")
    kid.update(real.to("cuda"), real=True)
    kid.update(to_uint8_batch(images).to("cuda"), real=False)
    kid_mean, kid_std = kid.compute()
    return {"kid_mean": round(float(kid_mean), 4), "kid_std": round(float(kid_std), 4),
            "clip_score": round(float(np.mean(clip_scores(images, list(validation["caption"])))), 2)}


MODELS = ["base", "run1_lora_r8", "run2_lora_r32"] + list(EXTRA_RUNS)

results = []
for model in MODELS:
    results.append({"model": model, "guidance": GUIDANCE, **generate_and_score(model, GUIDANCE, save=True)})
    print(results[-1])

validation_results = pd.DataFrame(results)
validation_results.to_csv(METRIC_DIR / "validation_results.csv", index=False)

In [ ]:
print(validation_results[["model", "kid_mean", "kid_std", "clip_score"]].to_string(index=False))

## 6. Is any extra run better than run 2?

The decision rule from the top of the notebook, applied to the numbers above.

In [ ]:
table = validation_results.set_index("model")
run2 = table.loc["run2_lora_r32"]

better = []
for name in EXTRA_RUNS:
    kid_better = table.loc[name, "kid_mean"] < run2["kid_mean"] - run2["kid_std"]
    clip_ok = table.loc[name, "clip_score"] >= run2["clip_score"]
    print(f"{name:18s} KID {table.loc[name, 'kid_mean']:.4f} (run 2: {run2['kid_mean']:.4f}) | "
          f"CLIP {table.loc[name, 'clip_score']:.2f} (run 2: {run2['clip_score']:.2f}) | "
          f"KID clearly lower: {kid_better} | CLIP not lower: {clip_ok}")
    if kid_better and clip_ok:
        better.append(name)

print("\nExtra runs that meet the rule:", better if better else "none, run 2 stays the final model")

## 7. Guidance scale

A higher guidance scale makes the sampler follow the prompt more strongly (usually a higher CLIP score, but less natural images). The base model, run 2 and the extra run with the lowest KID are generated at four values, again on the validation prompts only.

In [ ]:
best_extra = min(EXTRA_RUNS, key=lambda name: table.loc[name, "kid_mean"])
print("Extra run with the lowest validation KID:", best_extra)

sweep = []
for guidance in [5.0, 7.5, 9.0, 11.0]:
    for model in ["base", "run2_lora_r32", best_extra]:
        sweep.append({"model": model, "guidance": guidance, **generate_and_score(model, guidance)})
        print(sweep[-1])

guidance_sweep = pd.DataFrame(sweep)
guidance_sweep.to_csv(METRIC_DIR / "guidance_sweep.csv", index=False)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for model, group in guidance_sweep.groupby("model"):
    axes[0].plot(group["guidance"], group["clip_score"], marker="o", label=model)
    axes[1].plot(group["guidance"], group["kid_mean"], marker="o", label=model)
axes[0].set_title("CLIP score (higher is better)")
axes[1].set_title("KID (lower is better)")
for axis in axes:
    axis.set_xlabel("guidance scale")
    axis.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(METRIC_DIR / "guidance_sweep.png", dpi=120)
plt.show()

print(guidance_sweep.pivot(index="guidance", columns="model", values="clip_score").round(2).to_string())
print()
print(guidance_sweep.pivot(index="guidance", columns="model", values="kid_mean").round(4).to_string())

del pipe, clip
torch.cuda.empty_cache()